In [ ]:
# Task 1: Create all_image_paths list
import os
import random
import tensorflow as tf

base_dir = './images_dataSAT/'
class_0_dir = os.path.join(base_dir, 'class_0_non_agri')
class_1_dir = os.path.join(base_dir, 'class_1_agri')

class_0_paths = [os.path.join(class_0_dir, f) for f in os.listdir(class_0_dir) 
                 if f.endswith(('.png', '.jpg', '.jpeg'))]
class_1_paths = [os.path.join(class_1_dir, f) for f in os.listdir(class_1_dir) 
                 if f.endswith(('.png', '.jpg', '.jpeg'))]

all_image_paths = class_0_paths + class_1_paths


# Task 2: Create temp list with zip, randomly select and print 5
labels = [0] * len(class_0_paths) + [1] * len(class_1_paths)
temp = list(zip(all_image_paths, labels))
random.seed(42)
random.shuffle(temp)

for i in range(5):
    print(f"Image {i+1}: {temp[i][0]}, Label: {temp[i][1]}")


# Task 3: Generate batch of data (batch_size=8)
def custom_data_generator(image_paths, labels, batch_size=8, target_size=(224, 224)):
    while True:
        for i in range(0, len(image_paths), batch_size):
            batch_paths = image_paths[i:i+batch_size]
            batch_labels = labels[i:i+batch_size]
            batch_images = []
            for path in batch_paths:
                img = tf.keras.preprocessing.image.load_img(path, target_size=target_size)
                img = tf.keras.preprocessing.image.img_to_array(img) / 255.0
                batch_images.append(img)
            yield np.array(batch_images), np.array(batch_labels)

image_paths = [p for p, _ in temp]
image_labels = [l for _, l in temp]

batch_images, batch_labels = next(custom_data_generator(image_paths, image_labels, batch_size=8))
print(f"Batch images shape: {batch_images.shape}")
print(f"Batch labels shape: {batch_labels.shape}")


# Task 4: Create validation data (assuming validation split)
from sklearn.model_selection import train_test_split

train_paths, val_paths, train_labels, val_labels = train_test_split(
    image_paths, image_labels, test_size=0.2, random_state=42
)

val_generator = custom_data_generator(val_paths, val_labels, batch_size=8)
val_batch = next(val_generator)
print(f"Validation batch - Images: {val_batch[0].shape}, Labels: {val_batch[1].shape}")